# BreizhCrops: Benchmark Inspection

One sample is one Brittany parcel with a native Sentinel-2 L1C time series and a crop-type label. The benchmark uses the four published FRH regions and excludes Belle-Ile in the loader.


## Setup


In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

# Find repo root by walking upward until we see /src
REPO = Path.cwd()

while not (REPO / "src").exists() and REPO != REPO.parent:
    REPO = REPO.parent

# Make src imports work
sys.path.insert(0, str(REPO / "src"))

from dataio.get_input import get_input
from dataio.inspection import (
    benchmark_input_contract_table,
    benchmark_metadata_table,
)

BENCHMARK_ROOT = REPO / "data" / "input" / "benchmarks"

## Load Benchmark


In [2]:
MAX_SAMPLES = 5000
LOAD_KWARGS = {}

bench = None
load_error = None
try:
    bench = get_input("breizhcrops", root=BENCHMARK_ROOT, max_samples=MAX_SAMPLES, shuffle=False, **LOAD_KWARGS)
    print(f"loaded {bench.name}: {bench.n_samples} samples")
except Exception as exc:
    load_error = exc
    print(f"benchmark could not be loaded locally: {type(exc).__name__}: {exc}")

benchmark could not be loaded locally: OSError: [Errno 57] Can't synchronously read data (file read failed: time = Fri Jun 26 10:50:14 2026
, filename = '/Users/akshithchowdary/Developer/Projects/org/abe/robustness/data/input/benchmarks/breizhcrops/2017/L1C/frh01.h5', file descriptor = 86, errno = 57, error message = 'Socket is not connected', buf = 0x11904bc08, total read size = 6800, bytes this sub-read = 6800, offset = 38975720)


## Input Contract and Available Metadata


In [3]:
if bench is None:
    display(Markdown(f"Benchmark tables are skipped because the local data could not be read: `{type(load_error).__name__}: {load_error}`"))
else:
    display(Markdown("### Model-facing input contract"))
    display(benchmark_input_contract_table(bench))
    display(Markdown("### Other benchmark metadata available"))
    display(benchmark_metadata_table(bench))

Benchmark tables are skipped because the local data could not be read: `OSError: [Errno 57] Can't synchronously read data (file read failed: time = Fri Jun 26 10:50:14 2026
, filename = '/Users/akshithchowdary/Developer/Projects/org/abe/robustness/data/input/benchmarks/breizhcrops/2017/L1C/frh01.h5', file descriptor = 86, errno = 57, error message = 'Socket is not connected', buf = 0x11904bc08, total read size = 6800, bytes this sub-read = 6800, offset = 38975720)`

## Dataset Summary


In [4]:
if bench is not None and hasattr(bench, "native"):
    rows = []
    for modality in ("s2", "s1", "climate"):
        series = getattr(bench.native, modality)
        lengths = np.array([len(v) for v in series.values], dtype=int)
        rows.append({
            "modality": modality,
            "bands": len(series.bands),
            "band_names": ", ".join(series.bands) if series.bands else "-",
            "min_observations": int(lengths.min()) if len(lengths) else 0,
            "median_observations": float(np.median(lengths)) if len(lengths) else 0,
            "max_observations": int(lengths.max()) if len(lengths) else 0,
        })
    display(pd.DataFrame(rows))
else:
    print("native modality summary skipped")

native modality summary skipped


## Dataset Summary


In [5]:
if bench is not None and hasattr(bench, "labels"):
    labels = np.asarray(bench.labels)
    groups = np.asarray(bench.groups, dtype=object)
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    pd.Series(labels).value_counts().sort_index().plot(kind="bar", ax=ax[0], color="#2563eb")
    ax[0].set_title("label distribution")
    ax[0].set_xlabel("label")
    ax[0].set_ylabel("samples")
    pd.Series(groups).value_counts().sort_index().plot(kind="bar", ax=ax[1], color="#16a34a")
    ax[1].set_title("group distribution")
    ax[1].set_xlabel("group")
    ax[1].set_ylabel("samples")
    plt.tight_layout()
    plt.show()
else:
    print("sample-level label/group plot skipped")

sample-level label/group plot skipped


## Takeaways

BreizhCrops is S2-only and single-year. `groups` are NUTS-3 regions. Parcel coordinates are available when the package geometry can be read; otherwise the loader keeps coordinates as NaN without changing the sensor input contract.
